[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 04](README.md)

# MPI punto a punto y progreso

**Tema:** 04 · **Sesiones:** 16, 17, 18 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo hacer coincidir mensajes sin depender del orden accidental ni introducir interbloqueos?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** MPI obliga a hacer explícito quién envía, quién recibe y qué datos forman el mensaje. Esa precisión permite razonar sobre deadlock, progreso y halos.

**Prerrequisitos.**

- Procesos, memoria privada y paso de argumentos.
- Modelo de costo latencia–ancho de banda y referencia serial.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Identificar comunicador, rango, tag, datatype y count.
- Comparar bloqueo, no bloqueo y `MPI_Sendrecv`.
- Validar emparejamiento y vida útil de buffers.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

Un mensaje coincide por comunicador, origen permitido y tag; datatype/count determinan interpretación y capacidad.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

`MPI_Isend/Irecv` inicia operaciones cuyos buffers no pueden reutilizarse hasta completar la solicitud.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

El progreso y el buffering no deben usarse para justificar un patrón potencialmente bloqueante.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- rank — identidad de un proceso dentro de un comunicador
- tag — etiqueta que participa en el emparejamiento de mensajes
- colectiva — operación coordinada por todos los procesos del comunicador


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Mpi Comunicacion

![Procesos MPI con mensajes y colectiva](../../images/mpi-comunicacion.svg)

**Cómo leerlo.** Las flechas exteriores representan punto a punto; las interiores, coordinación colectiva. Todos los ranks deben respetar comunicador, orden y contrato de datos.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "04"
NOTEBOOK = "04_mpi/01_punto_a_punto.ipynb"
assert (ROOT / "curso" / "notebooks" / "04_mpi" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Anillo determinista

**Situación.** Se genera el contrato de mensajes para cada rango.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
def ring_contract(size, tag=17):
    return [{"rank": r, "send_to": (r+1)%size, "recv_from": (r-1)%size, "tag": tag} for r in range(size)]
contract = ring_contract(6)
for row in contract:
    sender = row["recv_from"]
    assert (sender + 1) % 6 == row["rank"]
    print(row)


### Explicación del resultado

El mismo tag es seguro dentro del contrato del anillo; fases distintas deben distinguirse o sincronizarse.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Descomposición de halo

**Situación.** Se calculan vecinos y rangos de un dominio 1D, incluidos extremos físicos.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
n, size = 25, 4
q, r = divmod(n, size)
start = 0
rows = []
for rank in range(size):
    local = q + (rank < r)
    rows.append((rank, start, start+local, rank-1 if rank else None, rank+1 if rank+1<size else None))
    start += local
assert rows[-1][2] == n
for row in rows: print(row)


### Lectura razonada

Los procesos extremos usan condiciones de frontera o `MPI_PROC_NULL`; no reciben un halo inexistente.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Qué contrato de source, destination, tag y count debe coincidir para que dos operaciones se emparejen?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Compilar `mpi/hello_mpi.c` y `mpi/ring_pass.c`.
2. Construir una variante `Sendrecv` y otra no bloqueante.
3. Probar tamaños de proceso 1, 2 y mayores que dos.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Asumir que `MPI_Send` siempre bufferiza.
- Reutilizar un buffer antes de Wait.
- Ignorar status y tamaño recibido.


## Criterios de aceptación

- Todos los mensajes tienen contrato compatible.
- No hay deadlock para tamaños admitidos.
- Errores MPI y códigos de salida se comprueban.


## Síntesis

- La pregunta que debes poder responder es: **¿Cómo hacer coincidir mensajes sin depender del orden accidental ni introducir interbloqueos?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Hello MPI](../../../mpi/hello_mpi.c)
- [Anillo](../../../mpi/ring_pass.c)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 04](README.md)
